# Discover species per genus from GTDB R226

**This notebook is NOT a Nextflow pipeline module.** It runs standalone, before
`nextflow run`, to produce a curated **per-genus species list** (one `<Genus>_species.txt`
file per target genus) plus an audit TSV. You then use those species lists to
construct the pipeline samplesheet by hand.

```
GTDB R226 + BV-BRC host DB  ──►  THIS NOTEBOOK  ──►  <Genus>_species.txt + expansion_audit.tsv
```

For 2–3 specific genera in a one-off analysis. The default pipeline behavior
(one arbitrary species per genus) stays untouched — this just gives you a
curated species panel to feed it.

**Method summary** (also printed in Cell 10 as a ready-to-paste paragraph):

1. Load a pre-downloaded GTDB R226 bacterial taxonomy file (download instructions in Cell 2).
2. For each target genus, enumerate all species in GTDB.
3. Filter to species that have a UniProt reference proteome (mirrors the
   pipeline's own query so positives here will succeed in `download_fasta.sh`).
4. Filter to species isolated from human hosts using a per-genus BV-BRC export
   (optional — genera without an entry in `HUMAN_HOST_DBS` pass through unfiltered).
5. Draw a seeded random sample of up to `MAX_PER_GENUS` species per genus —
   avoids alphabetical / quality bias of a sort-then-truncate. `MUST_INCLUDE`
   species are pre-populated and bypass the human-host filter.
6. Write per-genus species lists, an audit TSV, and a methods paragraph.

Cell asserts catch silent bugs at each transformation boundary.

## 1. Imports & config

In [36]:
import csv, gzip, hashlib, random, re, time, urllib.parse, urllib.request
from pathlib import Path
import pandas as pd

In [37]:
# --- GTDB release pinning ---
GTDB_RELEASE = "226.0"
GTDB_TAXONOMY = Path("/mnt/volume/workdir/projects/bact_peptides/db/gtdb_bac120_taxonomy_r226.tsv.gz")

# --- Per-analysis inputs (edit these) ---
AUDIT_LOG        = Path("expansion_audit.tsv")
GENERA_TO_EXPAND = ["Staphylococcus"]
MUST_INCLUDE     = {
    # Force specific species into a genus's selection. They count against
    # MAX_PER_GENUS — e.g. with cap=10 and one forced species, 9 more are
    # drawn randomly. Each forced species must have a UniProt reference
    # proteome, otherwise Cell 13 will assert. Forced species bypass the
    # human-host filter (a NOTE is printed if you force a non-human species).
    "Staphylococcus": ["Staphylococcus aureus"],
}
HUMAN_HOST_DBS   = {
    # Per-genus path to a BV-BRC genome export pre-filtered to human host.
    # Get exports from https://www.bv-brc.org/ → Genomes → filter by genus +
    # Host: Human → Download CSV. Only species listed in the file are kept
    # in the selection pool. Genera not listed here pass through unfiltered.
    "Staphylococcus": Path("/mnt/volume/workdir/projects/bact_peptides/db/2026-04-28_BVBRC_Staph_human.csv"),
}
MAX_PER_GENUS    = 10        # cap on total selected species per genus (forced + random)
RANDOM_SEED      = 42        # reruns are bit-identical with same seed

# --- Behavior toggles ---
COLLAPSE_GTDB_LINEAGES = True   # treat "Escherichia coli_A" as "Escherichia coli"
UNIPROT_THROTTLE_SEC   = 0.1    # sleep between UniProt REST calls

# --- Asserts on config ---
assert isinstance(GENERA_TO_EXPAND, list) and len(GENERA_TO_EXPAND) >= 1
assert all(isinstance(g, str) and " " not in g for g in GENERA_TO_EXPAND), (
    f"GENERA_TO_EXPAND must contain single-token genus names; got {GENERA_TO_EXPAND}"
)
assert MAX_PER_GENUS >= 1
assert isinstance(RANDOM_SEED, int)
assert isinstance(MUST_INCLUDE, dict)
for _g, _sps in MUST_INCLUDE.items():
    assert _g in GENERA_TO_EXPAND, (
        f"MUST_INCLUDE genus '{_g}' must also appear in GENERA_TO_EXPAND={GENERA_TO_EXPAND}"
    )
    assert isinstance(_sps, list) and all(isinstance(s, str) for s in _sps), (
        f"MUST_INCLUDE values must be lists of species strings; got {_sps!r} for '{_g}'"
    )
    assert all(s.startswith(f"{_g} ") for s in _sps), (
        f"MUST_INCLUDE species for '{_g}' must start with '{_g} ': {_sps}"
    )
    assert len(_sps) <= MAX_PER_GENUS, (
        f"MUST_INCLUDE for '{_g}' has {len(_sps)} entries, exceeds MAX_PER_GENUS={MAX_PER_GENUS}"
    )
assert isinstance(HUMAN_HOST_DBS, dict)
for _g, _p in HUMAN_HOST_DBS.items():
    assert isinstance(_p, Path), (
        f"HUMAN_HOST_DBS value for '{_g}' must be a Path; got {type(_p).__name__}"
    )

print(f"GTDB release      : {GTDB_RELEASE}")
print(f"Genera to expand  : {GENERA_TO_EXPAND}")
print(f"Must include      : {MUST_INCLUDE}")
print(f"Host filters      : {{{', '.join(f'{g!r}: {p.name!r}' for g, p in HUMAN_HOST_DBS.items())}}}")
print(f"Max per genus     : {MAX_PER_GENUS}")
print(f"Random seed       : {RANDOM_SEED}")
print(f"Collapse lineages : {COLLAPSE_GTDB_LINEAGES}")
print(f"GTDB taxonomy     : {GTDB_TAXONOMY}")
print(f"Audit log         : {AUDIT_LOG}")


GTDB release      : 226.0
Genera to expand  : ['Staphylococcus']
Must include      : {'Staphylococcus': ['Staphylococcus aureus']}
Host filters      : {'Staphylococcus': '2026-04-28_BVBRC_Staph_human.csv'}
Max per genus     : 10
Random seed       : 42
Collapse lineages : True
GTDB taxonomy     : /mnt/volume/workdir/projects/bact_peptides/db/gtdb_bac120_taxonomy_r226.tsv.gz
Audit log         : expansion_audit.tsv


## 2. Load GTDB taxonomy

This notebook does **not** download GTDB. It expects `bac120_taxonomy_r226.tsv.gz`
to already exist at the path set by `GTDB_TAXONOMY` in Cell 1
(default: `/mnt/volume/workdir/projects/bact_peptides/db/gtdb_bac120_taxonomy_r226.tsv.gz`).

If you need to (re)download it:

```bash
wget https://data.gtdb.ecogenomic.org/releases/release226/226.0/bac120_taxonomy_r226.tsv.gz
# or
curl -O https://data.gtdb.ecogenomic.org/releases/release226/226.0/bac120_taxonomy_r226.tsv.gz
```

SHA256 is computed and recorded in the audit log so an unexpected upstream change
would show up as a mismatch on rerun.

In [38]:
if not GTDB_TAXONOMY.exists():
    raise FileNotFoundError(
        f"GTDB taxonomy file not found at: {GTDB_TAXONOMY}\n\n"
        f"Download it first with:\n"
        f"  wget https://data.gtdb.ecogenomic.org/releases/release226/{GTDB_RELEASE}/bac120_taxonomy_r226.tsv.gz\n"
        f"or update GTDB_TAXONOMY in Cell 1 to point at an existing file."
    )

size_mb = GTDB_TAXONOMY.stat().st_size / 1024**2
print(f"Loaded GTDB taxonomy: {GTDB_TAXONOMY} ({size_mb:.1f} MB)")

# --- Asserts ---
assert GTDB_TAXONOMY.stat().st_size > 1_000_000, "GTDB file unexpectedly small"
with open(GTDB_TAXONOMY, "rb") as f:
    magic = f.read(2)
assert magic == b"\x1f\x8b", "GTDB file is not a valid gzip"

h = hashlib.sha256()
with open(GTDB_TAXONOMY, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        h.update(chunk)
GTDB_SHA256 = h.hexdigest()
print(f"SHA256: {GTDB_SHA256}")


Loaded GTDB taxonomy: /mnt/volume/workdir/projects/bact_peptides/db/gtdb_bac120_taxonomy_r226.tsv.gz (7.3 MB)
SHA256: a54d30b1deb505e813cb754f4298a225687df3cd882675cad7b53796d1429ebc


## 3. Parse GTDB into a DataFrame

GTDB taxonomy strings look like:
`d__Bacteria;p__Proteobacteria;c__...;g__Escherichia;s__Escherichia coli`.

GTDB sometimes splits a Linnaean species into sub-clusters (e.g.
`Escherichia coli_A`, `Escherichia coli_B`). With `COLLAPSE_GTDB_LINEAGES = True`
we strip the trailing `_<UPPER>` so these are treated as one species for
selection purposes — UniProt reference proteomes are keyed by Linnaean species
anyway, so the sub-cluster suffix would not survive downstream.

In [39]:
RE_GENUS   = re.compile(r"g__([^;]+)")
RE_SPECIES = re.compile(r"s__([^;]+)")
RE_LINEAGE_SUFFIX = re.compile(r"_[A-Z]+$")

rows = []
# dont want to store the unzipped table, easier to read in like this
with gzip.open(GTDB_TAXONOMY, "rt", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    for record in reader:
        if len(record) < 2:
            continue
        genome_id, tax = record[0], record[1]
        g = RE_GENUS.search(tax)
        s = RE_SPECIES.search(tax)
        if not g or not s:
            continue
        genus = g.group(1).strip()
        species_raw = s.group(1).strip()
        if not genus or not species_raw:
            continue
        rows.append((genome_id, genus, species_raw))

gtdb = pd.DataFrame(rows, columns=["genome_id", "genus", "species_raw"])

def _collapse(name: str) -> str:
    parts = name.rsplit(" ", 1)
    if len(parts) == 2:
        last_clean = RE_LINEAGE_SUFFIX.sub("", parts[1])
        return f"{parts[0]} {last_clean}"
    return RE_LINEAGE_SUFFIX.sub("", name)

gtdb["species_collapsed"] = gtdb["species_raw"].map(_collapse)
gtdb["species"] = gtdb["species_collapsed"] if COLLAPSE_GTDB_LINEAGES else gtdb["species_raw"]

print(f"Genomes                    : {len(gtdb):,}")
print(f"Unique genera              : {gtdb['genus'].nunique():,}")
print(f"Unique species (raw)       : {gtdb['species_raw'].nunique():,}")
print(f"Unique species (collapsed) : {gtdb['species_collapsed'].nunique():,}")
print(f"Active 'species' column    : {'collapsed' if COLLAPSE_GTDB_LINEAGES else 'raw'}")

# --- Asserts ---
assert len(gtdb) > 700_000, f"Unexpectedly few genomes: {len(gtdb)}"
assert gtdb["genus"].notna().all()
assert gtdb["species"].notna().all()
assert (gtdb["genus"] != "").all()
assert gtdb["species_raw"].nunique() > 100_000


Genomes                    : 715,230
Unique genera              : 27,326
Unique species (raw)       : 136,646
Unique species (collapsed) : 132,860
Active 'species' column    : collapsed


## 4. Per-genus candidate species (full list, no cap)

Eyeball this output. The next cell hits UniProt for each candidate; if you
spot lineages you definitely don't want included, you can either change
`COLLAPSE_GTDB_LINEAGES` and rerun from Cell 3, or just adjust `MAX_PER_GENUS`
or the random seed and accept the seeded sample.

In [40]:
candidates = {}  # genus -> sorted list of unique species
for genus in GENERA_TO_EXPAND:
    species_in_genus = sorted(gtdb.loc[gtdb["genus"] == genus, "species"].unique())
    candidates[genus] = species_in_genus
    print(f"\n=== {genus} ===")
    print(f"  {len(species_in_genus)} unique species in GTDB R{GTDB_RELEASE}")
    if not species_in_genus:
        print("  WARNING: no species found — check spelling against GTDB.")
        continue
    for sp in species_in_genus[:50]:
        print(f"    {sp}")
    if len(species_in_genus) > 50:
        print(f"    ... ({len(species_in_genus) - 50} more)")


=== Staphylococcus ===
  74 unique species in GTDB R226.0
    Staphylococcus agnetis
    Staphylococcus americanisciuri
    Staphylococcus argensis
    Staphylococcus argenteus
    Staphylococcus arlettae
    Staphylococcus aureus
    Staphylococcus auricularis
    Staphylococcus borealis
    Staphylococcus caeli
    Staphylococcus caledonicus
    Staphylococcus canis
    Staphylococcus capitis
    Staphylococcus caprae
    Staphylococcus carnosus
    Staphylococcus chromogenes
    Staphylococcus coagulans
    Staphylococcus cohnii
    Staphylococcus condimenti
    Staphylococcus cornubiensis
    Staphylococcus croceilyticus
    Staphylococcus debuckii
    Staphylococcus delphini
    Staphylococcus devriesei
    Staphylococcus durrellii
    Staphylococcus edaphicus
    Staphylococcus epidermidis
    Staphylococcus equorum
    Staphylococcus felis
    Staphylococcus gallinarum
    Staphylococcus haemolyticus
    Staphylococcus hominis
    Staphylococcus hsinchuensis
    Staphylococcus 

## 5. UniProt reference-proteome cross-check

Mirrors the pipeline's own query in `bin/download_fasta.sh`:
`taxonomy_name:<species_with_+>+AND+proteome_type:reference`. A `True` here
means the pipeline will successfully retrieve *something* for this name.

Note: `taxonomy_name` is fuzzy. A query for `Escherichia somespecies` may
match the genus and return an arbitrary E. coli proteome. That mirrors the
pipeline's existing behavior — what you see is what you get downstream.

In [41]:
UNIPROT_BASE = "https://rest.uniprot.org/proteomes/search"

def uniprot_has_reference_proteome(species: str):
    encoded = species.replace(" ", "+")
    url = (
        f"{UNIPROT_BASE}?query=taxonomy_name:{encoded}+AND+proteome_type:reference"
        f"&format=tsv&fields=upid&size=1"
    )
    try:
        with urllib.request.urlopen(url, timeout=30) as resp:
            status = resp.status
            body = resp.read().decode("utf-8")
    except Exception as e:
        print(f"  ERROR querying UniProt for '{species}': {e}")
        return (False, None, None)
    lines = [ln for ln in body.splitlines() if ln.strip()]
    if len(lines) < 2:
        return (False, None, status)
    upid = lines[1].split("\t")[0]
    return (True, upid, status)

records = []
for genus, species_list in candidates.items():
    print(f"\n=== {genus}: querying {len(species_list)} candidates ===")
    for sp in species_list:
        has, upid, status = uniprot_has_reference_proteome(sp)
        records.append({
            "genus": genus, "species": sp,
            "has_proteome": has, "upid": upid, "http_status": status,
        })
        time.sleep(UNIPROT_THROTTLE_SEC)
    n_yes = sum(1 for r in records if r["genus"] == genus and r["has_proteome"])
    print(f"  {n_yes} / {len(species_list)} have reference proteomes")

candidates_df = pd.DataFrame(records)

for genus in GENERA_TO_EXPAND:
    print(f"\n=== {genus} (annotated) ===")
    sub = candidates_df[candidates_df["genus"] == genus]
    print(sub.to_string(index=False))

# --- Asserts ---
assert len(candidates_df) == sum(len(v) for v in candidates.values())
for genus in GENERA_TO_EXPAND:
    n_yes = ((candidates_df["genus"] == genus) & candidates_df["has_proteome"]).sum()
    if candidates[genus] and n_yes == 0:
        print(f"WARNING: genus '{genus}' has 0 species with UniProt reference proteome")


=== Staphylococcus: querying 74 candidates ===
  61 / 74 have reference proteomes

=== Staphylococcus (annotated) ===
         genus                         species  has_proteome        upid  http_status
Staphylococcus          Staphylococcus agnetis          True UP000646308          200
Staphylococcus  Staphylococcus americanisciuri          True UP001205609          200
Staphylococcus         Staphylococcus argensis          True UP000242712          200
Staphylococcus        Staphylococcus argenteus          True UP000873311          200
Staphylococcus         Staphylococcus arlettae          True UP000006074          200
Staphylococcus           Staphylococcus aureus          True UP000305697          200
Staphylococcus      Staphylococcus auricularis          True UP000242470          200
Staphylococcus         Staphylococcus borealis          True UP000610527          200
Staphylococcus            Staphylococcus caeli          True UP000095768          200
Staphylococcus      S

## 6. Human-host filter (BV-BRC)

Restrict the candidate pool to species isolated from human hosts, using a
pre-downloaded BV-BRC genome export per genus. Genera without an entry in
`HUMAN_HOST_DBS` (Cell 1) pass through unfiltered. `MUST_INCLUDE` species
bypass this filter (a `NOTE` is printed in Cell 13 if you force a non-human
species).

The BV-BRC export must be **pre-filtered to human host** — the notebook
only checks that the file has a `Species` column, it does not re-validate
the host metadata. Get exports from <https://www.bv-brc.org/> → Genomes
→ filter by genus + Host: Human → Download CSV.

Adds a `human_associated` boolean column to `candidates_df`. Strain-only
entries in BV-BRC (e.g. `Staphylococcus sp. HMSC...`) won't match anything
in the GTDB-derived pool, so they are harmless.

In [42]:
HUMAN_HOST_SPECIES = {}  # genus -> frozenset of species names from BV-BRC
for genus, path in HUMAN_HOST_DBS.items():
    assert path.exists(), f"Human-host DB for '{genus}' not found: {path}"
    host_df = pd.read_csv(path, dtype=str, low_memory=False)
    assert "Species" in host_df.columns, (
        f"Expected 'Species' column in BV-BRC export {path}; got: {list(host_df.columns)[:10]}..."
    )
    # check that all human hosts
    assert (host_df["Host Common Name"] == "Human").all(), ('BV-BRC contains non-human hosts')
    species_set = frozenset(
        s.strip() for s in host_df["Species"].dropna().unique() if s.strip()
    )
    HUMAN_HOST_SPECIES[genus] = species_set
    print(f"  {genus}: {len(species_set)} unique species "
          f"(from {len(host_df):,} BV-BRC genomes in {path.name})")

# Annotate candidates_df. Genera without a host DB pass through unfiltered
# (human_associated=True for all their species).
def _is_human_associated(row):
    if row["genus"] not in HUMAN_HOST_DBS:
        return True
    return row["species"] in HUMAN_HOST_SPECIES[row["genus"]]

candidates_df["human_associated"] = candidates_df.apply(_is_human_associated, axis=1)

# Per-genus summary — show what got dropped so the filter is auditable
for genus in GENERA_TO_EXPAND:
    sub = candidates_df[candidates_df["genus"] == genus]
    if genus not in HUMAN_HOST_DBS:
        print(f"\n=== {genus} ===  (no host DB → no filter applied)")
        continue
    n_total = len(sub)
    n_uniprot = int(sub["has_proteome"].sum())
    n_human = int(sub["human_associated"].sum())
    n_pool = int((sub["has_proteome"] & sub["human_associated"]).sum())
    dropped = sorted(sub.loc[sub["has_proteome"] & ~sub["human_associated"], "species"].tolist())
    print(f"\n=== {genus} ===")
    print(f"  Candidates                       : {n_total}")
    print(f"  UniProt-positive                 : {n_uniprot}")
    print(f"  Human-associated (BV-BRC)        : {n_human}")
    print(f"  Pool (UniProt + human)           : {n_pool}")
    print(f"  Dropped (UniProt-pos, not human) : {len(dropped)}")
    for sp in dropped[:50]:
        print(f"    - {sp}")
    if len(dropped) > 50:
        print(f"    ... ({len(dropped) - 50} more)")

# --- Asserts ---
for genus in HUMAN_HOST_DBS:
    if genus not in GENERA_TO_EXPAND:
        print(f"WARNING: HUMAN_HOST_DBS includes '{genus}' but it's not in GENERA_TO_EXPAND — ignored.")
        continue
    assert len(HUMAN_HOST_SPECIES[genus]) > 0, f"Empty host species set for '{genus}'"
    n_pool = int(((candidates_df["genus"] == genus)
                  & candidates_df["has_proteome"]
                  & candidates_df["human_associated"]).sum())
    if n_pool == 0:
        print(f"WARNING: '{genus}' has 0 species that are both UniProt-positive and human-associated.")

  Staphylococcus: 291 unique species (from 22,308 BV-BRC genomes in 2026-04-28_BVBRC_Staph_human.csv)

=== Staphylococcus ===
  Candidates                       : 74
  UniProt-positive                 : 61
  Human-associated (BV-BRC)        : 43
  Pool (UniProt + human)           : 42
  Dropped (UniProt-pos, not human) : 19
    - Staphylococcus agnetis
    - Staphylococcus americanisciuri
    - Staphylococcus caeli
    - Staphylococcus canis
    - Staphylococcus carnosus
    - Staphylococcus delphini
    - Staphylococcus devriesei
    - Staphylococcus edaphicus
    - Staphylococcus hsinchuensis
    - Staphylococcus hyicus
    - Staphylococcus lloydii
    - Staphylococcus lutrae
    - Staphylococcus marylandisciuri
    - Staphylococcus microti
    - Staphylococcus muscae
    - Staphylococcus piscifermentans
    - Staphylococcus ratti
    - Staphylococcus rostri
    - Staphylococcus simiae


## 7. Seeded random selection

Pool is UniProt-positive species that are also human-associated (when a host
DB is provided for the genus). `MUST_INCLUDE` species are pre-populated and
**bypass** the human-host filter; the remaining `MAX_PER_GENUS - len(forced)`
slots are drawn at random. Per-genus deterministic RNG: seed is
`f"{RANDOM_SEED}:{genus}"`, so adding/removing a genus does not perturb the
picks for the others. The seed and `MAX_PER_GENUS` are recorded in the audit log.

In [43]:
SELECTED_SPECIES = {}
SELECTION_DETAIL = {}  # genus -> {"forced": [...], "random": [...]}

for genus in GENERA_TO_EXPAND:
    uniprot_pool = sorted(
        candidates_df.query("genus == @genus and has_proteome")["species"].tolist()
    )
    human_pool = sorted(
        candidates_df.query("genus == @genus and has_proteome and human_associated")["species"].tolist()
    )
    forced = sorted(MUST_INCLUDE.get(genus, []))

    # Forced species must have a UniProt reference proteome — otherwise
    # `download_fasta.sh` would fail on them. Fail loudly here.
    missing = [sp for sp in forced if sp not in uniprot_pool]
    assert not missing, (
        f"MUST_INCLUDE species not in UniProt-positive pool for '{genus}': {missing}. "
        f"Drop them from MUST_INCLUDE or confirm they have a UniProt reference proteome."
    )

    # Forced species bypass the human-host filter — but warn so it's not silent.
    if genus in HUMAN_HOST_DBS:
        non_human_forced = [sp for sp in forced if sp not in set(human_pool)]
        if non_human_forced:
            print(f"  NOTE: forced species not in human-host DB for '{genus}' "
                  f"(included anyway): {non_human_forced}")

    # Random picks come from the human pool (or full UniProt pool if no host DB), minus forced.
    random_source = human_pool  # = uniprot_pool when no host DB (human_associated=True for all)
    remaining_pool = [sp for sp in random_source if sp not in set(forced)]
    n_random = min(MAX_PER_GENUS - len(forced), len(remaining_pool))
    rng = random.Random(f"{RANDOM_SEED}:{genus}")
    random_picks = sorted(rng.sample(remaining_pool, n_random)) if n_random > 0 else []

    pick = sorted(set(forced) | set(random_picks))
    SELECTED_SPECIES[genus] = pick
    SELECTION_DETAIL[genus] = {"forced": forced, "random": random_picks}

    print(f"\n=== {genus} ===")
    print(f"  Pool (UniProt-positive)  : {len(uniprot_pool)}")
    print(f"  Pool (+ human-associated): {len(human_pool)}"
          + ("" if genus in HUMAN_HOST_DBS else "  (no host DB → same as UniProt pool)"))
    print(f"  Forced (MUST_INCLUDE)    : {len(forced)}")
    print(f"  Random picks             : {len(random_picks)}")
    print(f"  Total selected           : {len(pick)} (cap = {MAX_PER_GENUS})")
    print(f"  RNG seed                 : '{RANDOM_SEED}:{genus}'")
    forced_set = set(forced)
    for sp in pick:
        tag = "  [forced]" if sp in forced_set else ""
        print(f"    {sp}{tag}")

# --- Asserts ---
for genus in GENERA_TO_EXPAND:
    forced = set(MUST_INCLUDE.get(genus, []))
    human_pool = set(candidates_df.query(
        "genus == @genus and has_proteome and human_associated"
    )["species"].tolist())
    remaining = human_pool - forced
    expected = len(forced) + min(MAX_PER_GENUS - len(forced), len(remaining))
    assert len(SELECTED_SPECIES[genus]) == expected, (
        f"{genus}: selected {len(SELECTED_SPECIES[genus])}, expected {expected}"
    )
    assert len(SELECTED_SPECIES[genus]) == len(set(SELECTED_SPECIES[genus]))
    assert all(sp in SELECTED_SPECIES[genus] for sp in SELECTION_DETAIL[genus]["forced"]), (
        f"{genus}: a forced species is missing from the final selection"
    )

# Idempotency check — second draw must equal first
_check = {}
for genus in GENERA_TO_EXPAND:
    human_pool = sorted(candidates_df.query(
        "genus == @genus and has_proteome and human_associated"
    )["species"].tolist())
    forced = sorted(MUST_INCLUDE.get(genus, []))
    remaining_pool = [sp for sp in human_pool if sp not in set(forced)]
    n_random = min(MAX_PER_GENUS - len(forced), len(remaining_pool))
    rng = random.Random(f"{RANDOM_SEED}:{genus}")
    random_picks = sorted(rng.sample(remaining_pool, n_random)) if n_random > 0 else []
    _check[genus] = sorted(set(forced) | set(random_picks))
assert _check == SELECTED_SPECIES, "Random selection is not deterministic — seed bug"
print("\nIdempotency check: OK")


=== Staphylococcus ===
  Pool (UniProt-positive)  : 61
  Pool (+ human-associated): 42
  Forced (MUST_INCLUDE)    : 1
  Random picks             : 9
  Total selected           : 10 (cap = 10)
  RNG seed                 : '42:Staphylococcus'
    Staphylococcus aureus  [forced]
    Staphylococcus croceilyticus
    Staphylococcus debuckii
    Staphylococcus felis
    Staphylococcus nepalensis
    Staphylococcus saccharolyticus
    Staphylococcus saprophyticus
    Staphylococcus schleiferi
    Staphylococcus simulans
    Staphylococcus ureilyticus

Idempotency check: OK


## 8. Save selected species

In [45]:
output_files = {}
for genus, species in SELECTED_SPECIES.items():
    out_path = Path(f"{genus}_species.txt")
    with open(out_path, "w") as f:
        for sp in species:
            f.write(f"{sp}\n")
    output_files[genus] = out_path
    print(f"Wrote {len(species):>3} species → {out_path.resolve()}")

# --- Asserts ---
assert set(output_files) == set(GENERA_TO_EXPAND), (
    f"Output files don't cover all genera. Missing: "
    f"{set(GENERA_TO_EXPAND) - set(output_files)}"
)
for genus, path in output_files.items():
    assert path.exists() and path.stat().st_size > 0, f"Output missing or empty: {path}"
    with open(path) as f:
        lines = [ln.rstrip("\n") for ln in f if ln.strip()]
    assert len(lines) == len(SELECTED_SPECIES[genus]), (
        f"{path}: wrote {len(lines)} lines, expected {len(SELECTED_SPECIES[genus])}"
    )
    assert lines == SELECTED_SPECIES[genus], (
        f"{path} contents do not match SELECTED_SPECIES[{genus!r}]"
    )

Wrote  10 species → /mnt/volume/workdir/projects/bact_peptides/nf-core-mhcrefseq/bin/Staphylococcus_species.txt


## 9. Write audit log

In [46]:
audit_rows = []
for genus in GENERA_TO_EXPAND:
    sub = candidates_df[candidates_df["genus"] == genus]
    n_in_gtdb = int((gtdb["genus"] == genus).sum())
    n_uniprot = int(sub["has_proteome"].sum())
    n_human_uniprot = int((sub["has_proteome"] & sub["human_associated"]).sum())
    detail = SELECTION_DETAIL[genus]
    audit_rows.append({
        "genus": genus,
        "gtdb_release": GTDB_RELEASE,
        "gtdb_sha256": GTDB_SHA256,
        "host_db": str(HUMAN_HOST_DBS[genus]) if genus in HUMAN_HOST_DBS else "",
        "n_in_gtdb": n_in_gtdb,
        "n_with_uniprot_proteome": n_uniprot,
        "n_human_and_uniprot": n_human_uniprot,
        "n_forced": len(detail["forced"]),
        "n_random": len(detail["random"]),
        "n_selected": len(SELECTED_SPECIES[genus]),
        "max_per_genus": MAX_PER_GENUS,
        "random_seed": RANDOM_SEED,
        "forced_species": ";".join(detail["forced"]),
        "random_species": ";".join(detail["random"]),
        "selected_species": ";".join(SELECTED_SPECIES[genus]),
    })
audit = pd.DataFrame(audit_rows)
audit.to_csv(AUDIT_LOG, sep="\t", index=False)

print(f"Wrote audit log : {AUDIT_LOG.resolve()}")
print(f"Genera audited  : {len(audit_rows)}")
print(f"\nGTDB release : {GTDB_RELEASE}")
print(f"GTDB SHA256  : {GTDB_SHA256}")
print(f"Random seed  : {RANDOM_SEED}")

# --- Asserts ---
assert AUDIT_LOG.exists() and AUDIT_LOG.stat().st_size > 0
assert len(audit_rows) == len(GENERA_TO_EXPAND)


Wrote audit log : /mnt/volume/workdir/projects/bact_peptides/nf-core-mhcrefseq/bin/expansion_audit.tsv
Genera audited  : 1

GTDB release : 226.0
GTDB SHA256  : a54d30b1deb505e813cb754f4298a225687df3cd882675cad7b53796d1429ebc
Random seed  : 42


## 10. Methods-section snippet

In [ ]:
total_picked = sum(len(SELECTED_SPECIES[g]) for g in GENERA_TO_EXPAND)
total_forced = sum(len(SELECTION_DETAIL[g]["forced"]) for g in GENERA_TO_EXPAND)
filtered_genera = [g for g in GENERA_TO_EXPAND if g in HUMAN_HOST_DBS]

forced_clause = (
    f" Of these, {total_forced} were forcibly included via a curated MUST_INCLUDE list."
    if total_forced else ""
)
host_clause = (
    f" Candidate species were further restricted to those isolated from human "
    f"hosts according to the BV-BRC genome metadata (https://www.bv-brc.org/) "
    f"for the following genera: {', '.join(filtered_genera)}."
    if filtered_genera else ""
)

methods = f"""For samples annotated only at the genus level, a per-genus species panel was
constructed prior to running the nf-core/mhcrefseq pipeline. The Genome Taxonomy
Database release {GTDB_RELEASE} (RS226) was used as the source of bacterial
species concepts. For each target genus ({", ".join(GENERA_TO_EXPAND)}), all
species in the genus were enumerated and filtered to those with a UniProt
reference proteome (queried via the UniProt REST API with
`taxonomy_name:<species> AND proteome_type:reference`, mirroring the pipeline's
own download query).{host_clause} From the filtered pool, up to {MAX_PER_GENUS}
species per genus were drawn using a seeded random sample (Python
`random.Random` with seed `{RANDOM_SEED}:<genus>`) to avoid alphabetical or
quality-of-annotation bias. {total_picked} species were selected in total
across {len(GENERA_TO_EXPAND)} genera.{forced_clause}
The selected species were then used to construct the samplesheet for the
nf-core/mhcrefseq pipeline for proteome download, merging, and CD-HIT clustering."""

print(methods)
